<a href="https://colab.research.google.com/github/szabeebzaliz7-a11y/Datascience-And-Gen-AI/blob/main/Assignment_12_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Sentiment Analysis

Objective:
Develop machine learning models to classify emotions in text samples.

Dataset:

https://drive.google.com/file/d/1HWczIICsMpaL8EJyu48ZvRFcXx3_pcnb/view?usp=drive_link
Key components to be fulfilled :
1. Loading and Preprocessing (3 marks)
● Load the dataset and perform necessary preprocessing steps. This should include text
cleaning, tokenization, and removal of stopwords. Explain the preprocessing techniques
used and their impact on model performance.
2. Feature Extraction (2 marks):
● Implement feature extraction using CountVectorizer or TfidfVectorizer. Describe how the
chosen method transforms the text data into numerical features.
3. Model Development (2 marks):
● Train the following machine learning models
a)Naive Bayesb)Support Vector Machine

4. Model Comparison (2 marks)
● Evaluate the model using appropriate metrics (e.g., accuracy, F1-score). Provide a brief
explanation of the chosen model and its suitability for emotion classification.

In [13]:
import pandas as pd

# Load the dataset using the updated path
try:
    df = pd.read_csv(dataset_path)
    print("Dataset loaded successfully from: ", dataset_path)
except FileNotFoundError:
    print(f"Error: Dataset not found at {dataset_path}. Please ensure the file exists.")
    df = None

# Display the first 5 rows of the DataFrame and its information if loaded
if df is not None:
    display(df.head())
    print("\nDataset Info:")
    df.info()

Dataset loaded successfully from:  /content/nlp_dataset.csv


,Comment,Emotion
0,i seriously hate one subject to death but now ...,fear
1,im so full of life i feel appalled,anger
2,i sit here to write i start to dig out my feel...,fear
3,ive been really angry with r and i feel like a...,joy
4,i feel suspicious if there is no one outside l...,fear



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5937 entries, 0 to 5936
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Comment  5937 non-null   object
 1   Emotion  5937 non-null   object
dtypes: object(2)
memory usage: 92.9+ KB


### Text Preprocessing

Text preprocessing is a crucial step in Natural Language Processing (NLP) to prepare raw text data for machine learning models. It helps in transforming unstructured text into a more structured and analyzable format, which can significantly improve model performance.

We will perform the following preprocessing steps:

1.  **Text Cleaning:**
    *   **Lowercasing:** Converting all text to lowercase ensures that the model treats words like 'The' and 'the' as the same, reducing vocabulary size and improving consistency.
    *   **Removing Punctuation:** Punctuation marks often do not carry significant meaning for sentiment analysis and can be removed to reduce noise and the dimensionality of the feature space.
    *   **Removing Special Characters and Numbers:** Non-alphanumeric characters and numbers usually don't contribute to the sentiment of a text and can be removed to simplify the text.

2.  **Tokenization:**
    *   **Word Tokenization:** This process breaks down text into individual words or 'tokens'. These tokens are the basic units of analysis for most NLP models. This allows us to work with individual words rather than the entire sentence.

3.  **Stopword Removal:**
    *   **Removing Common Words:** Stopwords are common words (e.g., 'a', 'an', 'the', 'is', 'are') that appear frequently in a language but often carry little semantic meaning. Removing them can reduce noise, decrease feature space dimensionality, and help the model focus on more important, sentiment-bearing words. This can lead to faster training and potentially better performance.

These techniques collectively aim to reduce the complexity of the text data, standardize its representation, and highlight the most informative parts of the text for emotion classification.

In [14]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download necessary NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab') # Added to resolve LookupError

# Initialize NLTK stopwords
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # 1. Lowercasing
    text = text.lower()

    # 2. Removing punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Keep only letters and spaces

    # 3. Tokenization
    tokens = word_tokenize(text)

    # 4. Stopword Removal
    filtered_tokens = [word for word in tokens if word not in stop_words]

    # Join tokens back into a string
    return " ".join(filtered_tokens)

# Apply preprocessing to the 'Comment' column
if df is not None:
    print("Applying text preprocessing to the 'Comment' column...")
    df['cleaned_comment'] = df['Comment'].apply(preprocess_text)
    print("Preprocessing complete. Displaying the first 5 rows with cleaned comments:")
    display(df[['Comment', 'cleaned_comment', 'Emotion']].head())
else:
    print("DataFrame is empty, cannot perform preprocessing.")

Applying text preprocessing to the 'Comment' column...


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Preprocessing complete. Displaying the first 5 rows with cleaned comments:


,Comment,cleaned_comment,Emotion
0,i seriously hate one subject to death but now ...,seriously hate one subject death feel reluctan...,fear
1,im so full of life i feel appalled,im full life feel appalled,anger
2,i sit here to write i start to dig out my feel...,sit write start dig feelings think afraid acce...,fear
3,ive been really angry with r and i feel like a...,ive really angry r feel like idiot trusting fi...,joy
4,i feel suspicious if there is no one outside l...,feel suspicious one outside like rapture happe...,fear


## 2. Feature Extraction

Feature extraction is the process of converting raw text data into numerical features that can be understood and processed by machine learning algorithms. Text data, being unstructured, cannot be directly fed into models. Therefore, it needs to be transformed into a numerical representation.

We will use `TfidfVectorizer` for this purpose. Here's how it works and its impact:

*   **Term Frequency-Inverse Document Frequency (TF-IDF):**
    *   **Term Frequency (TF):** This measures how frequently a term (word) appears in a document. The more often a word appears in a document, the higher its TF score, indicating its importance within that specific document.
    *   **Inverse Document Frequency (IDF):** This measures how important a term is across the entire corpus. Words that are common across many documents (like stopwords, even if we removed most) get a lower IDF score, while unique or rare words that appear in only a few documents get a higher IDF score. This helps in down-weighting terms that appear very frequently and are therefore less informative.
    *   **TF-IDF Score:** The TF-IDF score is the product of TF and IDF. It increases proportionally to the number of times a word appears in the document but is offset by the frequency of the word in the corpus. This helps to highlight words that are unique and relevant to a particular document, making them good indicators for classification.

**Impact on Model Performance:**

`TfidfVectorizer` creates a matrix where each row represents a document (comment) and each column represents a unique word from the corpus. The values in the matrix are the TF-IDF scores for each word in each document. This numerical representation:

*   **Captures word importance:** It gives more weight to words that are significant in a specific comment but less common overall, which are often key indicators of emotion.
*   **Reduces dimensionality impact of common words:** By down-weighting common words, it helps the model focus on more distinctive features.
*   **Provides a structured input:** Machine learning models can directly use this numerical matrix as input for training, enabling them to learn patterns associated with different emotions.

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TfidfVectorizer
# You can experiment with parameters like max_features, ngram_range, min_df, max_df
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

# Fit and transform the 'cleaned_comment' data
if df is not None and 'cleaned_comment' in df.columns:
    X = tfidf_vectorizer.fit_transform(df['cleaned_comment'])
    y = df['Emotion'] # Target variable
    print("TF-IDF Vectorization complete.")
    print(f"Shape of the feature matrix (X): {X.shape}")
    print(f"Number of unique emotion labels (y): {y.nunique()}")
    print("First 5 feature names:\n", tfidf_vectorizer.get_feature_names_out()[:5])
else:
    print("DataFrame or 'cleaned_comment' column not found. Cannot perform feature extraction.")

TF-IDF Vectorization complete.
Shape of the feature matrix (X): (5937, 5000)
Number of unique emotion labels (y): 3
First 5 feature names:
 ['abandoned' 'abc' 'ability' 'abit' 'able']


In [16]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (4749, 5000)
Shape of X_test: (1188, 5000)
Shape of y_train: (4749,)
Shape of y_test: (1188,)


In [17]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Initialize and train the Multinomial Naive Bayes model
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_nb = nb_model.predict(X_test)

# Evaluate the Naive Bayes model
print("Naive Bayes Classifier Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_nb):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_nb))

Naive Bayes Classifier Performance:
Accuracy: 0.9175

Classification Report:
               precision    recall  f1-score   support

       anger       0.91      0.92      0.91       400
        fear       0.91      0.92      0.92       388
         joy       0.93      0.92      0.92       400

    accuracy                           0.92      1188
   macro avg       0.92      0.92      0.92      1188
weighted avg       0.92      0.92      0.92      1188



In [18]:
from sklearn.svm import LinearSVC

# Initialize and train the Linear SVM model
svm_model = LinearSVC(random_state=42)
svm_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_svm = svm_model.predict(X_test)

# Evaluate the SVM model
print("Support Vector Machine (LinearSVC) Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_svm))

Support Vector Machine (LinearSVC) Performance:
Accuracy: 0.9461

Classification Report:
               precision    recall  f1-score   support

       anger       0.95      0.94      0.94       400
        fear       0.95      0.94      0.94       388
         joy       0.94      0.96      0.95       400

    accuracy                           0.95      1188
   macro avg       0.95      0.95      0.95      1188
weighted avg       0.95      0.95      0.95      1188



## 4. Model Comparison

We have trained and evaluated two machine learning models for emotion classification: Naive Bayes (MultinomialNB) and Support Vector Machine (LinearSVC).

Let's summarize their performance based on the accuracy and F1-score:

*   **Naive Bayes (MultinomialNB):**
    *   Accuracy: `0.9175`
    *   Weighted Average F1-score: `0.92`

*   **Support Vector Machine (LinearSVC):**
    *   Accuracy: `0.9461`
    *   Weighted Average F1-score: `0.95`

### Analysis and Conclusion:

From the results, it's clear that the **Support Vector Machine (LinearSVC) model outperformed the Naive Bayes classifier** in this emotion classification task. The SVM achieved a higher accuracy of approximately `94.61%` and a weighted average F1-score of `0.95`, compared to the Naive Bayes' accuracy of `91.75%` and F1-score of `0.92`.

**Suitability for Emotion Classification:**

*   **Support Vector Machine (LinearSVC):** SVMs are generally very effective in high-dimensional spaces, which is characteristic of text classification problems where TF-IDF features can result in many features. They work by finding the optimal hyperplane that separates classes with the largest margin, which often leads to good generalization performance. The `LinearSVC` is particularly well-suited for large datasets with sparse features, making it an excellent choice for our text-based emotion classification.

*   **Naive Bayes (MultinomialNB):** While Naive Bayes models are simple, fast to train, and perform surprisingly well on text data due to their underlying probabilistic nature (especially with word counts or TF-IDF), their 'naive' assumption of feature independence can sometimes limit their performance compared to more complex models like SVMs. However, they are still a strong baseline for text classification.

**In conclusion, for this specific dataset and feature set, the Support Vector Machine (LinearSVC) is the more suitable model due to its superior performance in distinguishing between emotions.**